### 1:

In [10]:
import sqlite3 
import pandas as pd
db_ref = 'AdventureWorks.db'

In [2]:


queryQ1 = '''

SELECT MIN(p.ListPrice) AS min_price, MAX(p.ListPrice) AS max_price, MAX(p.ListPrice) - MIN(p.ListPrice) as price_diff, pc.ParentProductCategoryID, pc.Name, COUNT(*) AS 'type quantity'
FROM productcategory as pc
JOIN product as p
ON p.ProductCategoryID=pc.ProductCategoryID
GROUP BY pc.ParentProductCategoryID
'''

with sqlite3.connect(db_ref) as conn:
        price_ranges = pd.read_sql_query(queryQ1, conn)

price_ranges.head()

,min_price,max_price,price_diff,ParentProductCategoryID,Name,type quantity
0,539.99,3578.27,3038.28,1,Road Bikes,97
1,20.24,1431.50,1411.26,2,Road Frames,134
2,8.99,89.99,81.00,3,Bib-Shorts,35
3,2.29,159.00,156.71,4,Bike Stands,29


In [7]:
sql_query = '''
    SELECT ParentProductCategoryID,
        (SELECT name
        FROM ProductCategory
        WHERE (ProductCategoryID == pc.ParentProductCategoryID)
        ) as Name, MIN(p.ListPrice) AS min_price, MAX(p.ListPrice) AS max_price, MAX(p.ListPrice) - MIN(p.ListPrice) as price_diff, COUNT(*) AS 'type quantity'
    FROM ProductCategory AS pc
        JOIN Product AS p
        ON pc.ProductCategoryID = p.ProductCategoryID
    GROUP BY pc.ParentProductCategoryID
            '''

with sqlite3.connect(db_ref) as conn:
    df = pd.read_sql_query(sql_query, conn)
df.head()

,ParentProductCategoryID,Name,min_price,max_price,price_diff,type quantity
0,1,Bikes,539.99,3578.27,3038.28,97
1,2,Components,20.24,1431.50,1411.26,134
2,3,Clothing,8.99,89.99,81.00,35
3,4,Accessories,2.29,159.00,156.71,29


### 2. Price Ranges by Subcategory

In [4]:
from sympy import product

queryQ1 = '''

SELECT MIN(p.ListPrice) AS min_price, MAX(p.ListPrice) AS max_price, MAX(p.ListPrice) - MIN(p.ListPrice) as price_diff, pc.name, COUNT(*) AS 'type quantity'
FROM product as p
JOIN productcategory as pc
ON p.ProductCategoryID=pc.ProductCategoryID
GROUP BY p.ProductCategoryID
'''

with sqlite3.connect(db_ref) as conn:
        price_ranges = pd.read_sql_query(queryQ1, conn)

price_ranges.head()

,min_price,max_price,price_diff,Name,type quantity
0,539.99,3399.99,2860.00,Mountain Bikes,32
1,539.99,3578.27,3038.28,Road Bikes,43
2,742.35,2384.07,1641.72,Touring Bikes,22
3,44.54,120.27,75.73,Handlebars,8
4,53.99,121.49,67.50,Bottom Brackets,3


In [26]:
queryQ2 = '''
SELECT *
FROM product
'''
with sqlite3.connect(db_ref) as conn:
    price_basket = pd.read_sql_query(queryQ2, conn)


price_basket["Price Bracket"] = pd.cut(
    price_basket["ListPrice"],
    bins=[0, 50, 100, 500, 1000, float("inf")],
    labels=["Under $50","Under $100", "Under $500", "Under $1000", "Over $1000"],
    right=False
)

price_basket.sample(10)

,ProductID,Name,ProductNumber,Color,StandardCost,ListPrice,Size,Weight,ProductCategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,ThumbNailPhoto,ThumbnailPhotoFileName,rowguid,ModifiedDate,Price Bracket
12,717,"HL Road Frame - Red, 62",FR-R92R-62,Red,868.6342,1431.50,62,1043.26,18,6,2005-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,052E4F8B-0A2A-46B2-9F42-10FEBCFAE416,2008-03-11 10:01:36.827,Over $1000
94,799,"Road-550-W Yellow, 42",BK-R64Y-42,Yellow,713.0798,1120.49,42,8223.59,6,29,2006-07-01 00:00:00.000,,,47494638396150003100F70000E2E3E4EEE62C74777A34...,racer02_yellow_f_small.gif,A359AB09-16F2-4726-A60F-0DFDB1C9527E,2008-03-11 10:01:36.827,Over $1000
174,879,All-Purpose Bike Stand,ST-1401,,59.4660,159.00,,,31,122,2007-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,C7BB564B-A637-40F5-B21B-CBF7E4F713BE,2008-03-11 10:01:36.827,Under $500
271,976,"Road-350-W Yellow, 48",BK-R79Y-48,Yellow,1082.5100,1700.99,48,7447.95,6,27,2007-07-01 00:00:00.000,,,47494638396150003100F700000000080C040C10101018...,roadster_yellow_f_small.gif,EC4284DC-85FA-44A8-89EC-77FC9B71720A,2008-03-11 10:01:36.827,Over $1000
204,909,ML Mountain Seat/Saddle,SE-M798,,17.3782,39.14,,,19,80,2007-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,30222244-0FD8-4D28-8448-F2E658BF52BD,2008-03-11 10:01:36.827,Under $50
211,916,HL Touring Seat/Saddle,SE-T924,,23.3722,52.64,,,19,67,2007-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,0E158724-934D-4A64-991F-94FEC00BDEA8,2008-03-11 10:01:36.827,Under $100
132,837,"HL Road Frame - Black, 62",FR-R92B-62,Black,868.6342,1431.50,62,1043.26,18,6,2006-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,5DCE9C8C-FB94-46F8-B826-11658F6A3682,2008-03-11 10:01:36.827,Over $1000
22,727,"LL Road Frame - Red, 52",FR-R38R-52,Red,187.1571,337.22,52,1088.62,18,9,2005-07-01 00:00:00.000,2007-06-30 00:00:00.000,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,C455E0B3-D716-419D-ABF0-7E03EFDD2E26,2008-03-11 10:01:36.827,Under $500
6,711,"Sport-100 Helmet, Blue",HL-U509-B,Blue,13.0863,34.99,,,35,33,2005-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,FD7C0858-4179-48C2-865B-ABD5DFC7BC1D,2008-03-11 10:01:36.827,Under $50
100,805,LL Headset,HS-0296,,15.1848,34.20,,,15,59,2006-07-01 00:00:00.000,2007-06-30 00:00:00.000,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,BB6BD7B3-A34D-4D64-822E-781FA6838E19,2008-03-11 10:01:36.827,Under $50


In [27]:
queryQ2 = '''
SELECT
CASE
WHEN ListPrice < 100 THEN 'Cheap'
WHEN ListPrice BETWEEN 1000 AND 2000 THEN 'Medium'
ELSE 'Expensive'
END AS Price_Basket,
COUNT(*) AS Product_Count
FROM product
GROUP BY Price_Basket;
'''
with sqlite3.connect(db_ref) as conn:
    price_basket = pd.read_sql_query(queryQ2, conn)

price_basket.head()



,Price_Basket,Product_Count
0,Cheap,90
1,Expensive,154
2,Medium,51
